In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import joblib
import shap
import os


In [ ]:

df = pd.read_csv("data/BVBRC_genome_amr.zip", compression="zip")

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

print("✅ All libraries imported successfully.")

In [ ]:

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\n Column names:")
print(df.columns.tolist())
print("\n Data types:")
print(df.dtypes)
print("\n Null counts (top columns):")
print(df.isnull().sum().sort_values(ascending=False).head(10))


In [ ]:

print(" EXPLORATORY DATA ANALYSIS (EDA)")

print("\n Resistant Phenotype value counts (including NaN):")
print(df['Resistant Phenotype'].value_counts(dropna=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("EDA Overview", fontsize=15, fontweight='bold')


label_counts = df['Resistant Phenotype'].value_counts(dropna=False)
label_counts.index = label_counts.index.fillna('NaN (unlabeled)')
axes[0].bar(label_counts.index, label_counts.values,
            color=['#2ecc71', '#e74c3c', '#f39c12', '#bdc3c7', '#95a5a6'])
axes[0].set_title("Resistant Phenotype Distribution")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 300, f'{v:,}', ha='center', fontsize=9)

top_ab = df['Antibiotic'].value_counts().head(15)
axes[1].barh(top_ab.index[::-1], top_ab.values[::-1], color='steelblue')
axes[1].set_title("Top 15 Antibiotics")
axes[1].set_xlabel("Count")


ev_counts = df['Evidence'].value_counts(dropna=True)
axes[2].pie(ev_counts.values, labels=ev_counts.index, autopct='%1.1f%%',
            colors=['#3498db', '#e67e22'])
axes[2].set_title("Evidence Type")

plt.tight_layout()
plt.savefig('/content/eda_overview.png', bbox_inches='tight')
plt.show()
print("\nEDA chart saved.")

In [ ]:

print("DATA CLEANING")


raw_len = len(df)

df_clean = df.dropna(subset=['Resistant Phenotype']).copy()
print(f" After dropping unlabeled rows: {len(df_clean):,} (dropped {raw_len - len(df_clean):,})")


df_clean = df_clean[df_clean['Resistant Phenotype'].isin(['Resistant', 'Susceptible'])].copy()
print(f"  After removing Intermediate/Nonsusceptible: {len(df_clean):,}")

before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f" After de-duplication: {len(df_clean):,} (removed {before_dedup - len(df_clean):,} dups)")

df_clean['target'] = (df_clean['Resistant Phenotype'] == 'Resistant').astype(int)
print(f"\nFinal class balance:")
print(df_clean['target'].value_counts().rename({1: 'Resistant (1)', 0: 'Susceptible (0)'}))
print(f"\nClass ratio: {df_clean['target'].mean():.2%} Resistant")

In [ ]:
print("=" * 55)
print("            FEATURE ENGINEERING")
print("=" * 55)

fe = df_clean.copy()

# --- Feature 1: Extract Genus (first word of Genome Name) ---
fe['genus'] = fe['Genome Name'].str.split().str[0].str.lower()
top_genera = fe['genus'].value_counts().nlargest(20).index
fe['genus'] = fe['genus'].where(fe['genus'].isin(top_genera), other='other')

# --- Feature 2: Antibiotic (keep as-is, will label-encode) ---
top_antibiotics = fe['Antibiotic'].value_counts().nlargest(30).index
fe['antibiotic_clean'] = fe['Antibiotic'].where(fe['Antibiotic'].isin(top_antibiotics), other='other')

# --- Feature 3: Measurement Value (log-transform, fill NaN with median) ---
# Convert 'Measurement Value' to numeric, coercing errors to NaN
fe['measurement_val'] = pd.to_numeric(fe['Measurement Value'], errors='coerce')
med_val = fe['measurement_val'].median()
fe['measurement_val'] = fe['measurement_val'].fillna(med_val)
fe['measurement_val_log'] = np.log1p(fe['measurement_val'])
fe['measurement_val_missing'] = fe['Measurement Value'].isna().astype(int)  # missingness indicator

# --- Feature 4: Evidence type (Lab vs Computational) ---
fe['is_lab_evidence'] = (fe['Evidence'] == 'Laboratory Method').astype(int)

# --- Feature 5: Testing standard ---
fe['testing_std'] = fe['Testing Standard'].fillna('unknown')
fe['testing_std'] = fe['testing_std'].where(fe['testing_std'].isin(['CLSI', 'EUCAST']), other='other')

# --- Feature 6: Genus × Antibiotic interaction (frequency encoding) ---
fe['genus_antibiotic'] = fe['genus'] + '__' + fe['antibiotic_clean']
ga_resistance_rate = fe.groupby('genus_antibiotic')['target'].mean()
fe['ga_resistance_rate'] = fe['genus_antibiotic'].map(ga_resistance_rate)

# --- Feature 7: Per-antibiotic resistance rate ---
ab_resistance_rate = fe.groupby('antibiotic_clean')['target'].mean()
fe['ab_resistance_rate'] = fe['antibiotic_clean'].map(ab_resistance_rate)

# --- Feature 8: Per-genus resistance rate ---
genus_resistance_rate = fe.groupby('genus')['target'].mean()
fe['genus_resistance_rate'] = fe['genus'].map(genus_resistance_rate)

# --- Encode categorical features ---
le_genus = LabelEncoder()
le_ab = LabelEncoder()
le_std = LabelEncoder()

fe['genus_enc']   = le_genus.fit_transform(fe['genus'])
fe['ab_enc']      = le_ab.fit_transform(fe['antibiotic_clean'])
fe['std_enc']     = le_std.fit_transform(fe['testing_std'])

# --- Final feature set ---
FEATURES = [
    'genus_enc',
    'ab_enc',
    'std_enc',
    'measurement_val_log',
    'measurement_val_missing',
    'is_lab_evidence',
    'ga_resistance_rate',
    'ab_resistance_rate',
    'genus_resistance_rate',
]

TARGET = 'target'

X = fe[FEATURES]
y = fe[TARGET]

print(f"✅ Feature matrix shape: {X.shape}")
print(f"📋 Features used: {FEATURES}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Train size : {len(X_train):,}")
print(f"Test size  : {len(X_test):,}")
print(f"Train class balance: {y_train.mean():.2%} Resistant")
print(f"Test  class balance: {y_test.mean():.2%} Resistant")

In [ ]:
print(" MODEL TRAINING — XGBoost")


# Compute scale_pos_weight for imbalance handling
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f"scale_pos_weight: {spw:.2f}  (neg={neg:,} / pos={pos:,})")

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
    verbosity=0,
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print(f"XGBoost trained. Best iteration: {xgb_model.best_iteration}")

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train, y_train)
print("Random Forest trained.")


In [ ]:

print(" ENSEMBLE — Soft Voting")

# Build a lean XGB (no early stopping) for ensemble
xgb_for_ensemble = XGBClassifier(
    n_estimators=xgb_model.best_iteration + 1,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb_for_ensemble.fit(X_train, y_train)

ensemble = VotingClassifier(
    estimators=[
        ('xgb', xgb_for_ensemble),
        ('rf', rf_model),
    ],
    voting='soft',
    weights=[0.65, 0.35],
)
ensemble.fit(X_train, y_train)
print(" Ensemble model trained.")

In [ ]:

print("MODEL EVALUATION")

def evaluate_model(model, name, X_tr, y_tr, X_te, y_te):
    y_pred  = model.predict(X_te)
    y_prob  = model.predict_proba(X_te)[:, 1]

    acc  = accuracy_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred, average='weighted')
    f1b  = f1_score(y_te, y_pred, average='binary')
    auc  = roc_auc_score(y_te, y_prob)

    print(f"\n{'─'*45}")
    print(f"  Model : {name}")
    print(f"  Accuracy         : {acc:.4f}")
    print(f"  F1-score (weighted) : {f1:.4f}")
    print(f"  F1-score (binary)   : {f1b:.4f}")
    print(f"  AUC-ROC          : {auc:.4f}")
    print(f"{'─'*45}")
    print(classification_report(y_te, y_pred, target_names=['Susceptible', 'Resistant']))

    return y_pred, y_prob, {'name': name, 'accuracy': acc, 'f1_weighted': f1,
                             'f1_binary': f1b, 'auc_roc': auc}

_, _,  xgb_metrics   = evaluate_model(xgb_for_ensemble, "XGBoost",       X_train, y_train, X_test, y_test)
_, _,  rf_metrics    = evaluate_model(rf_model,          "Random Forest", X_train, y_train, X_test, y_test)
ens_pred, ens_prob, ens_metrics = evaluate_model(ensemble, "🏆 Ensemble", X_train, y_train, X_test, y_test)

In [ ]:

from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Ensemble Model, Evaluation Plots", fontweight='bold', fontsize=14)

# Confusion Matrix
cm = confusion_matrix(y_test, ens_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Susceptible', 'Resistant'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f"Confusion Matrix\nF1={ens_metrics['f1_weighted']:.4f} | AUC={ens_metrics['auc_roc']:.4f}")

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, ens_prob)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2.5,
             label=f'Ensemble AUC = {ens_metrics["auc_roc"]:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_title("ROC Curve")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/evaluation_plots.png', bbox_inches='tight')
plt.show()
print("Evaluation plots saved.")

In [ ]:


print("FEATURE IMPORTANCE")


importance_df = pd.DataFrame({
    'Feature': FEATURES,
    'XGBoost Importance': xgb_for_ensemble.feature_importances_,
    'RF Importance': rf_model.feature_importances_,
}).sort_values('XGBoost Importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Feature Importance Comparison", fontweight='bold', fontsize=14)

# XGBoost
axes[0].barh(importance_df['Feature'][::-1], importance_df['XGBoost Importance'][::-1],
             color='#e74c3c')
axes[0].set_title("XGBoost Feature Importance")
axes[0].set_xlabel("Importance Score")

# Random Forest
rf_sorted = importance_df.sort_values('RF Importance', ascending=False)
axes[1].barh(rf_sorted['Feature'][::-1], rf_sorted['RF Importance'][::-1],
             color='#3498db')
axes[1].set_title("Random Forest Feature Importance")
axes[1].set_xlabel("Importance Score")

plt.tight_layout()
plt.savefig('/content/feature_importance.png', bbox_inches='tight')
plt.show()

print("\n📊 Feature Importance Table:")
print(importance_df.to_string(index=False))

In [ ]:
MODEL_SAVE_PATH = '/content/drive/MyDrive/loopverse_amr_ensemble.joblib'
joblib.dump(ensemble, MODEL_SAVE_PATH)
print(f"Ensemble model saved to: {MODEL_SAVE_PATH}")

In [ ]:


print(" GENERATING PREDICTIONS CSV")


# Run predictions on the FULL test set
y_pred_final  = ensemble.predict(X_test)
y_prob_final  = ensemble.predict_proba(X_test)[:, 1]

# Build submission dataframe using original test indices
test_indices  = X_test.index
submission_df = fe.loc[test_indices, ['Genome ID', 'Antibiotic']].copy()
submission_df['Predicted_Phenotype'] = np.where(y_pred_final == 1, 'Resistant', 'Susceptible')
submission_df['Confidence_Score']    = y_prob_final.round(4)
submission_df = submission_df.reset_index(drop=True)
submission_df.columns = ['Genome_ID', 'Antibiotic', 'Predicted_Phenotype', 'Confidence_Score']

SUBMISSION_PATH = '/content/drive/MyDrive/predictions.csv'
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission CSV saved to: {SUBMISSION_PATH}")
print(f"   Rows in submission: {len(submission_df):,}")
print(f"\n Sample predictions:")
print(submission_df.head(10).to_string(index=False))

In [ ]:
 print()

print("     FINAL METRICS REPORT ")
print()

print(f"Model: Ensemble (XGBoost + Random Forest)")
print()

print(f"Total samples: {len(fe):,}")
print(f"Train set: {len(X_train):,}")
print(f"Test set: {len(X_test):,}")
print()

print("Test Performance:")
print(f"Accuracy  : {ens_metrics['accuracy']:.4f}")
print(f"F1-score  : {ens_metrics['f1_binary']:.4f}")
print(f"AUC-ROC   : {ens_metrics['auc_roc']:.4f}")
print()

print("Confusion Matrix:")
print(cm)
print()

print("Classification Report:")
print(classification_report(y_test, ens_pred, target_names=['Susceptible', 'Resistant']))
print()

print("Outputs saved:")
print(f"- Submission file: {SUBMISSION_PATH}")
print(f"- Model file     : {MODEL_SAVE_PATH}")